In [9]:
!pip install librosa soundfile numpy matplotlib scipy > /dev/null

In [10]:
"""
phase_vs_mgdf_experiment.py

Experimentos comparativos: magnitude, phase (wrapped/unwrapped), group delay (GD) e MGDF.
Gera: figuras PNG e arquivos WAV (original, phase-only, mag-only, swapped-phase).

Dependências:
  pip install librosa soundfile numpy matplotlib scipy
"""
import os
import numpy as np
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

# ========== parâmetros ==========
SR = 22050                # sample rate alvo
N_FFT = 2048
HOP = 512
WIN_LEN = N_FFT
GAMMA = 0.2               # parâmetro MGDF (Murty & Yegnanarayana)
ALPHA = 0.4               # parâmetro MGDF
SMOOTH_SIGMA = 1.0        # suavização espectral (em bins) para magnitude antes de MGDF
EPS = 1e-8

# arquivos de entrada
FILE1 = "voice01.wav"
FILE2 = "oi_lindo.wav"

# checagens iniciais
if not os.path.exists(FILE1) or not os.path.exists(FILE2):
    raise FileNotFoundError(f"Coloque '{FILE1}' e '{FILE2}' no diretório atual.")

# ========== utilitários ==========
def load_mono(path, sr=SR):
    x, _ = librosa.load(path, sr=sr, mono=True)
    # normalizar amplitude para evitar saturação de escrita
    x = x / (np.max(np.abs(x)) + 1e-12)
    return x

def pad_or_trim(x1, x2):
    L = min(len(x1), len(x2))
    return x1[:L], x2[:L]

def stft(x):
    return librosa.stft(x, n_fft=N_FFT, hop_length=HOP, win_length=WIN_LEN, window='hann')

def istft(X):
    return librosa.istft(X, hop_length=HOP, win_length=WIN_LEN, window='hann', length=None)

# ========== operações de fase / GD / MGDF ==========
def unwrap_phase(X):
    # X: complex STFT (freq_bins x frames)
    return np.unwrap(np.angle(X), axis=0)  # unwrapping along frequency axis

def compute_group_delay_from_spectrum(X):
    """
    Compute group delay per frame using frequency-derivative formula:
    tau(omega) = - d phi / d omega = (Re(X) * dIm(X)/dω - Im(X) * dRe(X)/dω) / |X|^2
    We approximate derivative wrt frequency by finite differences across frequency bins.
    Returns array same shape as X (freq x frames).
    """
    real = np.real(X)
    imag = np.imag(X)
    # derivative along frequency bins (axis=0)
    dreal = np.gradient(real, axis=0)
    dimag = np.gradient(imag, axis=0)
    denom = real**2 + imag**2 + EPS
    numer = real * dimag - imag * dreal
    gd = numer / denom
    # optional: negative sign convention depending on definition; we'll keep this form
    return gd

def smooth_magnitude(mag, sigma=SMOOTH_SIGMA):
    # smooth magnitude across frequency bins per frame
    sm = gaussian_filter1d(mag, sigma=sigma, axis=0, mode='reflect')
    return sm

def compute_mgdf(X, gamma=GAMMA, alpha=ALPHA, sigma=SMOOTH_SIGMA):
    """
    Compute Modified Group Delay Function (MGDF) per Murty & Yegnanarayana.
    Steps (practical approximation):
      - magnitude smoothing
      - raise smoothed magnitude to power gamma, multiply STFT -> Y = X * (smoothed_mag**gamma)
      - compute numerator using derivative of Y (as in GD formula)
      - denominator uses (smoothed_mag)^(2*alpha)
    Returns MGDF matrix (freq x frames).
    """
    mag = np.abs(X)
    smag = smooth_magnitude(mag, sigma=sigma) + EPS
    weight = smag ** gamma
    Y = X * weight
    # derivatives of Y
    Y_real = np.real(Y)
    Y_imag = np.imag(Y)
    dY_real = np.gradient(Y_real, axis=0)
    dY_imag = np.gradient(Y_imag, axis=0)
    numer = Y_real * dY_imag - Y_imag * dY_real
    denom = (smag ** (2 * alpha)) + EPS
    mgdf = numer / denom
    return mgdf

# ========== carregar sinais ==========
x1 = load_mono(FILE1, sr=SR)
x2 = load_mono(FILE2, sr=SR)

# alinhar comprimentos
x1, x2 = pad_or_trim(x1, x2)

# aplicar leve preênfase (opcional, pode realçar excitação glotal)
pre_emph = 0.97
x1 = np.append(x1[0], x1[1:] - pre_emph * x1[:-1])
x2 = np.append(x2[0], x2[1:] - pre_emph * x2[:-1])

# ========== STFTs ==========
S1 = stft(x1)
S2 = stft(x2)
mag1 = np.abs(S1)
mag2 = np.abs(S2)
phase1 = np.angle(S1)
phase2 = np.angle(S2)
unwrapped1 = unwrap_phase(S1)
unwrapped2 = unwrap_phase(S2)

# ========== GD e MGDF ==========
GD1 = compute_group_delay_from_spectrum(S1)
GD2 = compute_group_delay_from_spectrum(S2)

MGDF1 = compute_mgdf(S1, gamma=GAMMA, alpha=ALPHA, sigma=SMOOTH_SIGMA)
MGDF2 = compute_mgdf(S2, gamma=GAMMA, alpha=ALPHA, sigma=SMOOTH_SIGMA)

# normalizar para plots (melhor visualização)
def norm01(A):
    A = np.array(A, dtype=float)
    A = A - np.nanmin(A)
    if np.nanmax(A) > 0:
        A = A / (np.nanmax(A) + 1e-12)
    return A

# ========== reconstruções áudio (experimentais) ==========
# 1) original (reconstruct to ensure consistency)
y1_orig = istft(S1)
y2_orig = istft(S2)
# 2) phase-only (unity magnitude + original phase)
unity_mag = np.ones_like(S1)
y1_phase_only = istft(unity_mag * np.exp(1j * phase1))
y2_phase_only = istft(unity_mag * np.exp(1j * phase2))
# 3) mag-only (original magnitude + zero phase)
y1_mag_only = istft(mag1 * np.exp(1j * np.zeros_like(phase1)))
# 4) swapped-phase (magnitude of 1 + phase of 2)
# ensure shapes match (they do because we aligned signals)
y_swap = istft(mag1 * np.exp(1j * phase2))

# salvar WAVs (normalizar p/ evitar clipping)
def save_wav(x, path, sr=SR):
    x = x / (np.max(np.abs(x)) + 1e-12) * 0.95
    sf.write(path, x, sr)

os.makedirs("outputs", exist_ok=True)
save_wav(y1_orig, "outputs/voz1_original.wav")
save_wav(y2_orig, "outputs/voz2_original.wav")
save_wav(y1_phase_only, "outputs/voz1_phase_only.wav")
save_wav(y2_phase_only, "outputs/voz2_phase_only.wav")
save_wav(y1_mag_only, "outputs/voz1_mag_only.wav")
save_wav(y_swap, "outputs/voz1_mag_with_voz2_phase.wav")

# ========== plots comparativos ==========
import matplotlib
matplotlib.use('Agg')  # headless safe

freqs = np.linspace(0, SR/2, N_FFT//2 + 1)
frame_idx = S1.shape[1] // 4  # escolher um quadro para plot 1D (terceira parte)

# 1) espectrogramas (magnitude)
plt.figure(figsize=(12, 6))
plt.subplot(2,2,1)
plt.title("Magnitude (voz1) - espectrograma (dB)")
librosa.display.specshow(librosa.amplitude_to_db(mag1, ref=np.max), sr=SR, hop_length=HOP, y_axis='linear', x_axis='time')
plt.colorbar(format='%+2.0f dB')

plt.subplot(2,2,2)
plt.title("Magnitude (voz2) - espectrograma (dB)")
librosa.display.specshow(librosa.amplitude_to_db(mag2, ref=np.max), sr=SR, hop_length=HOP, y_axis='linear', x_axis='time')
plt.colorbar(format='%+2.0f dB')

# 2) unwrapped phase slice vs
plt.subplot(2,2,3)
plt.title("Unwrapped phase (freq slice) - quadro idx {}".format(frame_idx))
plt.plot(freqs, unwrapped1[:, frame_idx], label='voz1')
plt.plot(freqs, unwrapped2[:, frame_idx], label='voz2', alpha=0.8)
plt.xlabel("Hz")
plt.legend()

# 3) MGDF images
plt.tight_layout()
plt.savefig("outputs/magnitude_and_phase_slice.png", dpi=200)

# plot GD and MGDF as images for visual comparação
plt.figure(figsize=(12,6))
plt.subplot(2,2,1)
plt.title("Group Delay (voz1) - image (normalized)")
plt.imshow(norm01(GD1), origin='lower', aspect='auto', extent=[0, x1.shape[0]/SR, 0, SR/2])
plt.colorbar()
plt.subplot(2,2,2)
plt.title("Group Delay (voz2) - image (normalized)")
plt.imshow(norm01(GD2), origin='lower', aspect='auto', extent=[0, x2.shape[0]/SR, 0, SR/2])
plt.colorbar()
plt.subplot(2,2,3)
plt.title("MGDF (voz1) - image (normalized)")
plt.imshow(norm01(MGDF1), origin='lower', aspect='auto', extent=[0, x1.shape[0]/SR, 0, SR/2])
plt.colorbar()
plt.subplot(2,2,4)
plt.title("MGDF (voz2) - image (normalized)")
plt.imshow(norm01(MGDF2), origin='lower', aspect='auto', extent=[0, x2.shape[0]/SR, 0, SR/2])
plt.colorbar()
plt.tight_layout()
plt.savefig("outputs/gd_mgdf_images.png", dpi=200)

# plot 1D slices to inspect differences at chosen frame
plt.figure(figsize=(10,6))
plt.subplot(2,1,1)
plt.title("MGDF - quadro idx {}".format(frame_idx))
plt.plot(freqs, norm01(MGDF1[:, frame_idx]), label='voz1 MGDF')
plt.plot(freqs, norm01(MGDF2[:, frame_idx]), label='voz2 MGDF', alpha=0.8)
plt.legend()
plt.subplot(2,1,2)
plt.title("Group Delay (same quadro)")
plt.plot(freqs, norm01(GD1[:, frame_idx]), label='voz1 GD')
plt.plot(freqs, norm01(GD2[:, frame_idx]), label='voz2 GD', alpha=0.8)
plt.legend()
plt.tight_layout()
plt.savefig("outputs/mgdf_gd_slices.png", dpi=200)

print("===== experiment complete =====")
print("WAVs salvos em: outputs/*.wav")
print("Figuras salvas em: outputs/*.png")


===== experiment complete =====
WAVs salvos em: outputs/*.wav
Figuras salvas em: outputs/*.png
